# Defensive line WHILE PRESSING -- distance of the back line from goal (real clips + 2D animations)

**Every number here is computed only on pressing frames**: frames inside a press sequence of the team (Step 2,
`pressure/press_sequences.parquet`). Not while attacking, not in a passive defending block, not at set pieces.
Counter-presses (pressing in the first seconds after losing the ball, still in the attacking shape) are excluded by
default (`INCLUDE_COUNTER_PRESS`).

The question answered: **when a team presses, how high does its back line hold behind the press?** (does the back
line step up with the pressers, or stay deep and leave space between the lines?)

Per pressing frame: the **last defender** (deepest back-line player) distance from own goal = line height, flatness,
width, goalkeeper -> line, line -> midfield, ball position. Back line inferred from the defending shape (no names).
Then: team summary (overall, per pressing zone, per pressing situation), line height vs ball position while pressing,
one point per press over the half, step-up / drop events during presses, per defender, and automatically chosen
**real clips** (line through the defenders, last defender marked, live graph, mini-pitch) and **2D animations**.

In [ ]:
import os, io, json, pickle
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle

CACHE_DIR = r"C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\barca_atletico_first_half\new_cache"
VIDEO_PATH = r"C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\barca_atletico_first_half\clip.mkv"
PLAYER_TABLE = os.path.join(CACHE_DIR, "player_frame_table_named.parquet")
BALL_TABLE   = os.path.join(CACHE_DIR, "ball_frame_table_named.parquet")
META_PATH    = os.path.join(CACHE_DIR, "match_meta_named.json")
HOMOGRAPHY_PATH = os.path.join(CACHE_DIR, "homography_cache_barca_atletico_firsthalf.pkl")
OUT_DIR  = os.path.join(CACHE_DIR, "defensive_line")
PRESS_DIR = os.path.join(CACHE_DIR, "pressure")                 # Step 2 output (press_sequences.parquet)
INCLUDE_COUNTER_PRESS = False    # counter-presses happen in the attacking shape -> excluded
PRESS_PAD_S = 0.0                # extend each press sequence by this many seconds on both sides
CLIP_DIR = os.path.join(OUT_DIR, "clips")
ANIM_DIR = os.path.join(OUT_DIR, "animations")

TEAM_COLORS = {"BAR": (152, 77, 0), "ATM": (36, 53, 203)}     # BGR (clips); figures use the same colours
LINE_GAP_M = 5.0                 # line detection from average depths (same rule as the pressing-structure notebook)
MIN_DEF_VISIBLE = 3              # line height only when >= 3 back-line players are on screen
SMOOTH_FRAMES = 13               # rolling median for events / timeline
EVENT_WINDOW_S = 2.0             # drop / step-up = line moves >= EVENT_MIN_M within 2 s
EVENT_MIN_M = 8.0
EVENT_SEPARATION_S = 5.0
CLIP_PRE_S, CLIP_POST_S = 3.0, 3.0
RENDER_CLIPS, RENDER_ANIMATIONS = True, True
ANIM_STEP = 2                    # 2D animation uses every 2nd frame (12.5 fps)
PX_PER_METER = 10.0

for d in (OUT_DIR, CLIP_DIR, ANIM_DIR):
    os.makedirs(d, exist_ok=True)
pf = pd.read_parquet(PLAYER_TABLE)
bt = pd.read_parquet(BALL_TABLE).set_index("frame_idx").sort_index()
meta = json.load(open(META_PATH, encoding="utf-8"))
FPS, L, W, N_FRAMES = meta["fps"], meta["pitch_length"], meta["pitch_width"], meta["n_frames"]
TEAM_NAMES = {int(k): v for k, v in meta["team_names"].items()}
ATTACK_DIR = {int(k): v for k, v in meta["attack_dir"].items()}
PEOPLE = {int(k): v for k, v in meta["people"].items()}
NUM = {t: v["number"] for t, v in PEOPLE.items()}
LABEL = {t: f"{v['team_name'] or ''} {v['number'] if v['number'] is not None else ''} {v['name']}".strip() for t, v in PEOPLE.items()}
RGB = {nm: tuple(c / 255 for c in col[::-1]) for nm, col in TEAM_COLORS.items()}


def along_x(x, team): return x if ATTACK_DIR[team] == 1 else L - x
def along_y(y, team): return y if ATTACK_DIR[team] == 1 else W - y
def to_pitch_x(x_att, team): return x_att if ATTACK_DIR[team] == 1 else L - x_att
def mmss(f): return f"{int(f / FPS // 60):02d}:{f / FPS % 60:04.1f}"


live = pf[~pf.is_stoppage & pf.pitch_x.notna()].copy()
live["y_att"] = np.where(live.attack_dir.astype(float) == 1, live.pitch_y, W - live.pitch_y)
print(f"live rows {len(live):,} | teams {TEAM_NAMES} | attack dir {ATTACK_DIR}")

## 1. Back line of each team (the deepest players of the average defending shape)

In [ ]:
# ---- Section 1: back line of each team = the deepest outfield players on average OUT of possession ----
# (in possession the full-backs push up, so the back line is read from the defending shape)
BACK_LINE_N = 4      # 4 = back four (3 or 5 for a back three / five)
MID_LINE_N = 4       # the next players by depth = midfield line (for the line -> midfield gap)

DEF, MID, GK = {}, {}, {}
for k, nm in TEAM_NAMES.items():
    t = live[live.team == k]
    defending = t[(t.possession_team.notna() & (t.possession_team != k)).fillna(False)]
    src = defending if defending.frame_idx.nunique() >= 60 * FPS else t      # fall back to all frames if too little
    g = src.groupby("track_id").agg(role=("role", "first"), x=("x_att", "mean"), y=("y_att", "mean"),
                                    n=("frame_idx", "size"))
    g = g[g.n >= 5 * FPS]
    out = g[g.role == "player"].sort_values("x")
    DEF[k] = [int(i) for i in out.index[:BACK_LINE_N]]
    MID[k] = [int(i) for i in out.index[BACK_LINE_N:BACK_LINE_N + MID_LINE_N]]
    GK[k] = [int(i) for i in g[g.role == "goalkeeper"].index]
    print(f"{nm}: back line -> " + ", ".join(f"{LABEL[i]} ({out.loc[i, 'x']:.1f} m)"
                                             for i in sorted(DEF[k], key=lambda i: out.loc[i, 'y'])))
    print(f"   midfield line -> " + ", ".join(LABEL[i] for i in MID[k]))
    vis = t[t.track_id.isin(DEF[k])].groupby("frame_idx").size()
    print(f"   frames with >= {MIN_DEF_VISIBLE} back-line players visible: {(vis >= MIN_DEF_VISIBLE).sum() / FPS / 60:.1f} min "
          f"(of {t.frame_idx.nunique() / FPS / 60:.1f} min with the team on screen)")

## 2. Per-frame line metrics -- PRESSING frames only
A team-frame is kept only when: >= `MIN_DEF_VISIBLE` back-line players are visible, it is **inside a press sequence of
the team** (Step 2), it is **not a corner / box set piece**, it is **not a counter-press** (unless
`INCLUDE_COUNTER_PRESS`), and the deepest outfield player is a back-line player.
`line_x` = distance from own goal of the **last** (deepest) back-line defender.

In [ ]:
# ---- Section 2: per-frame line metrics -- only real, settled defending ----
CORNER_GAP_S = 3.0           # ball dead >= 3 s before a restart
CORNER_ZONE_M = 6.0          # restart taken within 6 m of a corner flag = corner
CORNER_PHASE_S = 10.0        # frames up to 10 s after the corner kick are removed
SET_PIECE_WINDOW_S = 20.0    # a box crowd within 20 s after a dead ball = set piece
BOX_DEF_MIN, BOX_ATT_MIN = 5, 3
SET_PIECE_TAIL_S = 3.0
REQUIRE_BACK_LINE_LAST = True
BOX_DEPTH, BOX_HALF_W = 16.5, 20.16

bx = bt.ball_pitch_x.reindex(range(N_FRAMES)).values
by = bt.ball_pitch_y.reindex(range(N_FRAMES)).values
poss = bt.possession_team.astype("float").reindex(range(N_FRAMES)).values
carr = bt.carrier_track_id.astype("float").reindex(range(N_FRAMES)).values
has = ~np.isnan(carr)

# ---- dead balls: runs with no carrier for >= CORNER_GAP_S ----
DEAD = []                                                    # (first dead frame, first frame with a carrier again)
for f in np.where(has[1:] & ~has[:-1])[0] + 1:
    b = f - 1
    while b >= 0 and not has[b]:
        b -= 1
    if (f - b - 1) >= CORNER_GAP_S * FPS:
        DEAD.append((b + 1, int(f)))
dead_end = np.array([e for _, e in DEAD], dtype=int)
dead_start = np.array([s for s, _ in DEAD], dtype=int)
frames_all = np.arange(N_FRAMES)
i_last = np.searchsorted(dead_end, frames_all, side="right") - 1          # last restart at or before each frame
since_restart = np.where(i_last >= 0, frames_all - dead_end[np.clip(i_last, 0, None)], 10 ** 9)

# ---- 1) corners: restart near a corner flag ----
corner_mask = np.zeros(N_FRAMES, dtype=bool)
CORNERS = []
for s0, f in DEAD:
    xy = next(((bx[j], by[j]) for j in range(f, min(f + 6, N_FRAMES)) if np.isfinite(bx[j])), None)
    if xy is None:
        r = live[(live.frame_idx == f) & (live.track_id == int(carr[f]))]
        xy = (r.pitch_x.iloc[0], r.pitch_y.iloc[0]) if len(r) else None
    if xy is None:
        continue
    if (xy[0] <= CORNER_ZONE_M or xy[0] >= L - CORNER_ZONE_M) and (xy[1] <= CORNER_ZONE_M or xy[1] >= W - CORNER_ZONE_M):
        e = min(N_FRAMES - 1, f + int(CORNER_PHASE_S * FPS))
        corner_mask[s0:e + 1] = True
        CORNERS.append({"kick_frame": int(f), "time": mmss(f), "from": int(s0), "to": int(e),
                        "taken_by": TEAM_NAMES.get(int(poss[f])) if np.isfinite(poss[f]) else None})

# ---- 2) box set pieces: crowd in a penalty area shortly after a dead ball (per defending team) ----
setpiece_mask = {k: np.zeros(N_FRAMES, dtype=bool) for k in TEAM_NAMES}
SETPIECES = []
pp = pf[pf.pitch_x.notna() & pf.role.isin(["player", "goalkeeper"]) & pf.team.notna()][["frame_idx", "team", "pitch_x", "pitch_y"]]
for k, nm in TEAM_NAMES.items():
    in_box = ((pp.pitch_x <= BOX_DEPTH) if ATTACK_DIR[k] == 1 else (pp.pitch_x >= L - BOX_DEPTH)) & \
             ((pp.pitch_y - W / 2).abs() <= BOX_HALF_W)
    cnt = pp[in_box].groupby(["frame_idx", "team"]).size().unstack(fill_value=0)
    n_def_box = cnt.get(k, pd.Series(0, index=cnt.index)).reindex(frames_all, fill_value=0).values
    n_att_box = cnt.get(1 - k, pd.Series(0, index=cnt.index)).reindex(frames_all, fill_value=0).values
    crowd = (n_def_box >= BOX_DEF_MIN) & (n_att_box >= BOX_ATT_MIN) & (since_restart <= SET_PIECE_WINDOW_S * FPS)
    if crowd.any():
        idx = np.where(crowd)[0]
        runs = np.split(idx, np.where(np.diff(idx) > FPS)[0] + 1)
        for r in runs:
            j = i_last[r[0]]
            s0 = int(dead_start[j]) if j >= 0 else int(r[0])
            e = min(N_FRAMES - 1, int(r[-1] + SET_PIECE_TAIL_S * FPS))
            setpiece_mask[k][s0:e + 1] = True
            SETPIECES.append({"defending": nm, "time": mmss(r[0]), "from": s0, "to": e})

# ---- 3) pressing frames: inside a press sequence of the team (Step 2) ----
_sit = os.path.join(PRESS_DIR, "team_structure", "press_sequences_situations.parquet")
SEQ = pd.read_parquet(_sit if os.path.exists(_sit) else os.path.join(PRESS_DIR, "press_sequences.parquet"))
press_seq = {k: np.full(N_FRAMES, -1, dtype=int) for k in TEAM_NAMES}
press_cp = {k: np.zeros(N_FRAMES, dtype=bool) for k in TEAM_NAMES}
_pad = int(PRESS_PAD_S * FPS)
for sid, r in SEQ.iterrows():
    k = int(r.team); a_, b_ = max(0, int(r.start) - _pad), min(N_FRAMES - 1, int(r.end) + _pad)
    press_seq[k][a_:b_ + 1] = int(sid)
    if bool(r.counter_press):
        press_cp[k][a_:b_ + 1] = True
SITUATION = SEQ["situation"].to_dict() if "situation" in SEQ.columns else {}
print(f"press sequences: {len(SEQ)} ({'with' if SITUATION else 'without'} situations from the pressing-structure notebook)")

print(f"dead balls: {len(DEAD)} | corners (flag rule): {len(CORNERS)} | box set pieces (crowd rule): {len(SETPIECES)}")
for c in CORNERS:
    print(f"   corner     {c['time']}  for {c['taken_by']}")
for c in SETPIECES:
    print(f"   set piece  {c['time']}  {c['defending']} defending")

# ---- line metrics per frame ----
recs = []
for k, nm in TEAM_NAMES.items():
    t = live[live.team == k]
    d = t[t.track_id.isin(DEF[k])].groupby("frame_idx").agg(n_def=("x_att", "size"), line_mean_x=("x_att", "mean"),
                                                           line_x=("x_att", "min"),          # line height = LAST defender
                                                           flatness=("x_att", "std"), y_min=("y_att", "min"), y_max=("y_att", "max"))
    deep = t[t.role == "player"].sort_values("x_att").groupby("frame_idx").head(1).set_index("frame_idx")
    m = t[t.track_id.isin(MID[k])].groupby("frame_idx").x_att.mean().rename("mid_x")
    g = t[t.track_id.isin(GK[k])].groupby("frame_idx").x_att.mean().rename("gk_x")
    df = d.join([deep.x_att.rename("last_defender_x"), deep.track_id.rename("last_defender_id"), m, g])
    df["width"] = df.y_max - df.y_min
    df["team"], df["team_name"] = k, nm
    df["last_is_back_line"] = df.last_defender_id.isin(DEF[k])
    df = df.reset_index()
    df["set_piece"] = setpiece_mask[k][df.frame_idx.values] | corner_mask[df.frame_idx.values]
    df["press_seq"] = press_seq[k][df.frame_idx.values]
    df["counter_press"] = press_cp[k][df.frame_idx.values]
    recs.append(df)
LH = pd.concat(recs, ignore_index=True)

LH["phase"] = np.where(LH.press_seq >= 0, "pressing", "not_pressing")
n0 = len(LH)
steps = [("< %d back-line players visible" % MIN_DEF_VISIBLE, LH.n_def >= MIN_DEF_VISIBLE),
         ("not pressing (attacking, defending without pressing, dead ball)", LH.phase == "pressing"),
         ("corner / box set piece", ~LH.set_piece)]
if not INCLUDE_COUNTER_PRESS:
    steps.append(("counter-press (attacking shape)", ~LH.counter_press))
if REQUIRE_BACK_LINE_LAST:
    steps.append(("deepest player not a defender", LH.last_is_back_line))
keep = np.ones(len(LH), dtype=bool)
for name, cond in steps:
    cond = np.asarray(cond, dtype=bool)
    print(f"   removed {int((keep & ~cond).sum()):>7,} team-frames: {name}")
    keep &= cond
LH = LH[keep].copy()
print(f"team-frames: {n0:,} -> {len(LH):,} kept")

LH["ball_x_att"] = [along_x(bx[f], k) if np.isfinite(bx[f]) else np.nan for f, k in zip(LH.frame_idx, LH.team)]
LH["block"] = np.select([LH.ball_x_att >= 70, LH.ball_x_att >= 35, LH.ball_x_att < 35], ["high", "mid", "low"], None)
LH["press_situation"] = LH.press_seq.map(SITUATION) if SITUATION else np.where(LH.counter_press, "counter_press", "press")
LH["gk_to_line_m"] = LH.line_x - LH.gk_x
LH["line_to_mid_m"] = LH.mid_x - LH.line_x
LH = LH.sort_values(["team", "frame_idx"]).reset_index(drop=True)
# smoothing in TIME (frames), never across removed frames
LH["line_x_smooth"] = np.nan
for k in TEAM_NAMES:
    sel = LH.team == k
    s = pd.Series(LH.loc[sel, "line_x"].values, index=LH.loc[sel, "frame_idx"].values).reindex(frames_all)
    sm = s.rolling(SMOOTH_FRAMES, center=True, min_periods=3).median()
    LH.loc[sel, "line_x_smooth"] = sm.reindex(LH.loc[sel, "frame_idx"].values).values
print("pressing frames with a measurable line: " + ", ".join(
    f"{nm} {(LH.team == k).sum() / FPS / 60:.1f} min in {LH[LH.team == k].press_seq.nunique()} presses"
    for k, nm in TEAM_NAMES.items()))

## 3. Team summary -- back line while pressing (overall, per pressing zone, per situation)

In [ ]:
def q(s): return f"{s.median():.1f} ({s.quantile(.25):.1f}-{s.quantile(.75):.1f})" if len(s) else "-"


def summ(g):
    return {"presses": g.press_seq.nunique(), "minutes": round(len(g) / FPS / 60, 1),
            "line height (last def) m, median (IQR)": q(g.line_x), "back-line mean m": round(g.line_mean_x.median(), 1),
            "flatness m": round(g.flatness.median(), 1), "width m": round(g.width.median(), 1),
            "GK -> line m": round(g.gk_to_line_m.median(), 1), "line -> midfield m": round(g.line_to_mid_m.median(), 1)}


prs = LH                                                    # every row is a pressing frame
summary = pd.DataFrame([{"team": nm, **summ(g)} for nm, g in prs.groupby("team_name")])
print("WHILE PRESSING:"); print(summary.to_string(index=False))
by_zone = pd.DataFrame([{"team": nm, "press zone (ball)": z, **summ(g)}
                        for (nm, z), g in prs.groupby(["team_name", "block"])])
print("\nby where the press happens (ball high / mid / low, seen from the pressing team):")
print(by_zone.to_string(index=False))
by_sit = pd.DataFrame([{"team": nm, "situation": z, **summ(g)} for (nm, z), g in prs.groupby(["team_name", "press_situation"])])
print("\nby pressing situation:"); print(by_sit.to_string(index=False))

## 4. Figures (pressing only): distribution, line height vs ball position, one point per press

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 5))
for nm in TEAM_NAMES.values():
    d = prs[prs.team_name == nm]
    axs[0].hist(d.line_x, bins=np.arange(0, 70, 2), alpha=0.5, color=RGB.get(nm), label=f"{nm} (median {d.line_x.median():.1f} m)")
    g = d.assign(bin=(d.ball_x_att // 5) * 5 + 2.5).groupby("bin").line_x
    med, lo, hi = g.median(), g.quantile(.25), g.quantile(.75)
    ok = g.size() >= FPS
    axs[1].plot(med[ok].index, med[ok], "-o", color=RGB.get(nm), label=nm)
    axs[1].fill_between(med[ok].index, lo[ok], hi[ok], color=RGB.get(nm), alpha=0.15)
axs[0].set_xlabel("last defender, m from own goal"); axs[0].legend()
axs[0].set_title("While pressing: back-line height")
axs[1].set_xlabel("ball position seen from the pressing team (m from own goal)"); axs[1].set_ylabel("last defender (m)")
axs[1].set_title("Does the back line follow the press? (median + IQR)"); axs[1].legend()
for xx in (35, 70): axs[1].axvline(xx, color="grey", ls=":", lw=0.8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "press_line_height_distribution_vs_ball.png"), dpi=110); plt.show()

per_press = prs.groupby(["team_name", "press_seq"]).agg(f=("frame_idx", "min"), line=("line_x", "median"),
                                                        ball=("ball_x_att", "median"), n=("frame_idx", "size")).reset_index()
fig, ax = plt.subplots(figsize=(18, 4))
for nm in TEAM_NAMES.values():
    d = per_press[per_press.team_name == nm]
    ax.scatter(d.f / FPS / 60, d.line, s=10 + d.n / 2, color=RGB.get(nm), alpha=0.7, label=f"{nm} (one dot = one press)")
ax.set_xlabel("match minute"); ax.set_ylabel("last defender, m from own goal"); ax.set_ylim(0, 60); ax.legend(loc="upper right")
ax.set_title("Back-line height during each press over the half (dot size = press length)")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "press_line_height_timeline.png"), dpi=110); plt.show()

## 5. Step-up / drop events DURING presses (line moves >= 8 m in 2 s)
`with_ball` = the ball moved the same way by a similar distance (the line follows a long ball / a pass forward);
`without_ball` = the line moved on its own (a run in behind, or pushing up / an offside trap).

In [ ]:
EV = []
win, sep = int(EVENT_WINDOW_S * FPS), int(EVENT_SEPARATION_S * FPS)
for k, nm in TEAM_NAMES.items():
    s = LH[(LH.team == k)].set_index("frame_idx")
    lx = s.line_x_smooth.reindex(range(N_FRAMES)); ph = s.phase.reindex(range(N_FRAMES))
    bxa = pd.Series([along_x(v, k) if np.isfinite(v) else np.nan for v in bx])
    delta = lx.shift(-win) - lx
    for kind, sign in (("drop", -1), ("step_up", 1)):
        cand = delta[(sign * delta >= EVENT_MIN_M) & (ph == "pressing")]
        chosen = []
        for f, v in cand.abs().sort_values(ascending=False).items():
            if all(abs(f - c) > sep for c in chosen):
                chosen.append(f)
                db = bxa.get(f + win, np.nan) - bxa.get(f, np.nan)
                EV.append({"team": k, "team_name": nm, "kind": kind, "start": int(f), "end": int(f + win),
                           "from_m": round(float(lx[f]), 1), "to_m": round(float(lx[f + win]), 1),
                           "change_m": round(float(delta[f]), 1), "ball_change_m": round(float(db), 1) if np.isfinite(db) else None,
                           "cause": ("with_ball" if np.isfinite(db) and np.sign(db) == sign and abs(db) >= 0.5 * abs(delta[f])
                                     else "without_ball"), "time": mmss(f)})
EV = pd.DataFrame(EV).sort_values(["team_name", "kind", "change_m"]) if EV else pd.DataFrame()
if len(EV):
    oop_min = prs.groupby("team_name").size() / FPS / 60
    print(EV.groupby(["team_name", "kind", "cause"]).size().unstack(fill_value=0).to_string())
    print("\nper 10 min of pressing:", {f"{nm} {kd}": round(len(g) / oop_min[nm] * 10, 1)
                                              for (nm, kd), g in EV.groupby(["team_name", "kind"])})
    print(EV.head(12).to_string(index=False))

## 6. Per defender WHILE PRESSING: depth, deviation from the line, how often he is the last man

In [ ]:
rows = []
for k, nm in TEAM_NAMES.items():
    pfr_k = set(LH[LH.team == k].frame_idx)                     # this team's pressing frames
    t = live[(live.team == k) & (live.role == "player") & live.frame_idx.isin(pfr_k)]
    deepest = t.loc[t.groupby("frame_idx").x_att.idxmin(), ["frame_idx", "track_id"]]
    lh = LH[(LH.team == k)].set_index("frame_idx").line_x
    for tid in DEF[k]:
        r = t[t.track_id == tid]
        rows.append({"team": nm, "defender": LABEL[tid], "frames": len(r),
                     "mean_depth_m": r.x_att.mean(), "mean_width_pos_m": r.y_att.mean(),
                     "vs_line_m": (r.x_att.values - lh.reindex(r.frame_idx).values).__array__().astype(float),
                     "last_man_share": (deepest.track_id == tid).sum() / max(len(deepest), 1)})
DT = pd.DataFrame(rows)
DT["vs_line_m"] = DT.vs_line_m.map(lambda a: float(np.nanmedian(a)) if len(a) else np.nan)
print(DT.round(2).to_string(index=False))

fig, axs = plt.subplots(1, len(TEAM_NAMES), figsize=(8 * len(TEAM_NAMES), 5))
for ax, (k, nm) in zip(np.atleast_1d(axs), TEAM_NAMES.items()):
    ax.add_patch(Rectangle((0, 0), L, W, fill=False)); ax.axvline(52.5, color="grey", lw=0.8)
    d = DT[DT.team == nm]
    med = prs[prs.team_name == nm].line_x.median()
    ax.axvline(med, color=RGB.get(nm), ls="--", lw=1.5, label=f"median last defender {med:.1f} m")
    for _, r in d.iterrows():
        tid = [t for t in DEF[k] if LABEL[t] == r.defender][0]
        ax.scatter(r.mean_depth_m, r.mean_width_pos_m, s=600, color=RGB.get(nm), edgecolor="white", zorder=3)
        ax.text(r.mean_depth_m, r.mean_width_pos_m, str(NUM.get(tid, "")), color="white", ha="center", va="center", weight="bold", zorder=4)
        ax.text(r.mean_depth_m, r.mean_width_pos_m + 4.5, f"last man {r.last_man_share:.0%}", ha="center", fontsize=8)
    ax.set_xlim(-2, 60); ax.set_ylim(W + 2, -2); ax.set_aspect("equal"); ax.legend(loc="lower right")
    ax.set_title(f"{nm} back line while pressing (own goal on the left)")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "press_back_line_defenders.png"), dpi=110); plt.show()

## 7. Moments for clips (chosen automatically)
All during presses. Per team: highest line, deepest line, a typical high press, biggest step-up, biggest drop.

In [ ]:
MOMENTS = []
for k, nm in TEAM_NAMES.items():
    o = prs[prs.team == k].dropna(subset=["line_x_smooth"])
    if len(o) == 0:
        continue
    hi_, lo_ = o.loc[o.line_x_smooth.idxmax()], o.loc[o.line_x_smooth.idxmin()]
    MOMENTS.append({"team": k, "team_name": nm, "kind": "highest_line", "start": int(hi_.frame_idx), "end": int(hi_.frame_idx),
                    "note": f"while pressing: last defender {hi_.line_x_smooth:.1f} m from goal"})
    MOMENTS.append({"team": k, "team_name": nm, "kind": "deepest_line", "start": int(lo_.frame_idx), "end": int(lo_.frame_idx),
                    "note": f"while pressing: last defender {lo_.line_x_smooth:.1f} m from goal"})
    hb = o[o.block == "high"]
    if len(hb):
        r = hb.iloc[(hb.line_x_smooth - hb.line_x_smooth.median()).abs().argsort().iloc[0]]
        MOMENTS.append({"team": k, "team_name": nm, "kind": "typical_high_press", "start": int(r.frame_idx), "end": int(r.frame_idx),
                        "note": f"line {r.line_x_smooth:.1f} m (team median when pressing high)"})
    if len(EV):
        for kd in ("drop", "step_up"):
            e = EV[(EV.team == k) & (EV.kind == kd)]
            if len(e):
                e = e.loc[e.change_m.abs().idxmax()]
                MOMENTS.append({"team": k, "team_name": nm, "kind": f"biggest_{kd}", "start": int(e.start), "end": int(e.end),
                                "note": f"{e.from_m} -> {e.to_m} m in {EVENT_WINDOW_S:.0f} s ({e.cause.replace('_', ' ')})"})
for m in MOMENTS:
    print(f"  {m['team_name']} {m['kind']:<20} {mmss(m['start'])}  {m['note']}")

## 8. Real match clips: line + offside line on the grass, back line highlighted, live graph, mini-pitch

In [ ]:
def _load_pickle(p):
    class _C(pickle.Unpickler):
        def find_class(self, m, n):
            return super().find_class(m.replace("numpy._core", "numpy.core", 1) if m.startswith("numpy._core") else m, n)
    b = open(p, "rb").read()
    try:
        return pickle.loads(b)
    except ModuleNotFoundError:
        return _C(io.BytesIO(b)).load()


def pitch_line_img(x_pitch, Hinv, Wv, Hv):
    pts = np.stack([np.full(35, x_pitch), np.linspace(0, W, 35)], 1) * PX_PER_METER
    ip = cv2.perspectiveTransform(pts[None].astype(np.float32), Hinv).reshape(-1, 2)
    ok = (ip[:, 0] > -Wv) & (ip[:, 0] < 2 * Wv) & (ip[:, 1] > -Hv) & (ip[:, 1] < 2 * Hv)
    return ip[ok].astype(np.int32)


def render_clip(m, HC, cap, fps_v):
    """Back line drawn as a line THROUGH the defenders' feet (left -> right), the last defender marked, and a
    dashed straight line across the pitch at the last defender's depth (= the line height / offside line)."""
    k, nm = m["team"], m["team_name"]
    s0, s1 = max(0, int(m["start"] - CLIP_PRE_S * FPS)), min(N_FRAMES - 1, int(m["end"] + CLIP_POST_S * FPS))
    rows = {f: g for f, g in pf[(pf.frame_idx >= s0) & (pf.frame_idx <= s1)].groupby("frame_idx")}
    lh = LH[(LH.team == k)].set_index("frame_idx")
    series = lh.line_x_smooth.reindex(range(s0, s1 + 1))
    cap.set(cv2.CAP_PROP_POS_FRAMES, s0)
    Wv, Hv = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)); sc = Hv / 1080
    path = os.path.join(CLIP_DIR, f"{nm}_{m['kind']}_{m['start']}.mp4")
    out = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps_v, (Wv, Hv))
    col = TEAM_COLORS.get(nm, (0, 0, 255))
    gw, gh = int(360 * sc), int(120 * sc); gx, gy = Wv - gw - int(15 * sc), int(55 * sc)
    mw, mh = int(315 * sc), int(204 * sc); msc = mw / L
    for f in range(s0, s1 + 1):
        ok, img = cap.read()
        if not ok:
            break
        H = HC[f] if isinstance(HC, list) and f < len(HC) else None
        Hinv = np.linalg.inv(np.asarray(H, np.float64)) if H is not None else None
        g = rows.get(f)
        lx = lh.line_x.get(f, np.nan)                                     # last defender, this frame
        defs = None
        if g is not None:
            defs = g[g.track_id.isin(DEF[k]) & g.pitch_x.notna()].copy()
            defs["xa"] = along_x(defs.pitch_x.values, k)
            defs["ya"] = along_y(defs.pitch_y.values, k)
            defs = defs.sort_values("ya")
        if not np.isfinite(lx):                  # not a pressing frame (attacking, passive defending, set piece)
            defs = None                          # -> no line at all
        last_id = int(defs.loc[defs.xa.idxmin(), "track_id"]) if defs is not None and len(defs) else None
        if Hinv is not None and np.isfinite(lx):                         # straight dashed line at the last defender
            q_ = pitch_line_img(to_pitch_x(lx, k), Hinv, Wv, Hv)
            for a_, b_ in zip(q_[::2], q_[1::2]):
                cv2.line(img, tuple(map(int, a_)), tuple(map(int, b_)), (0, 255, 255), max(1, int(2 * sc)), cv2.LINE_AA)
        if defs is not None and len(defs) >= 2:                          # line through the defenders' feet
            pts = np.stack([((defs.x1 + defs.x2) / 2).values, defs.y2.values], 1).astype(np.int32)
            cv2.polylines(img, [pts], False, (20, 20, 20), max(4, int(7 * sc)), cv2.LINE_AA)
            cv2.polylines(img, [pts], False, col, max(2, int(4 * sc)), cv2.LINE_AA)
        if g is not None:
            for t in g.itertuples():
                fx, fy = int((t.x1 + t.x2) / 2), int(t.y2)
                is_def = t.track_id in DEF[k]
                mine = (not pd.isna(t.team)) and int(t.team) == k
                c_ = col if mine else ((200, 200, 200) if t.role != "referee" else (30, 30, 30))
                rx = int(max((t.x2 - t.x1) * 0.6, 8 * sc))
                cv2.ellipse(img, (fx, fy), (rx, max(3, rx // 3)), 0, 0, 360, c_, int((4 if is_def else 2) * sc) or 1, cv2.LINE_AA)
                if t.track_id == last_id:                                  # the last defender
                    cv2.ellipse(img, (fx, fy), (rx + int(6 * sc), max(3, rx // 3) + int(4 * sc)), 0, 0, 360,
                                (0, 255, 255), max(2, int(3 * sc)), cv2.LINE_AA)
                if NUM.get(t.track_id) is not None:
                    tx = str(NUM[t.track_id]); (tw, th), _ = cv2.getTextSize(tx, cv2.FONT_HERSHEY_SIMPLEX, 0.5 * sc, 1)
                    cv2.rectangle(img, (fx - tw // 2 - 3, int(t.y1) - th - 9), (fx + tw // 2 + 3, int(t.y1) - 3), c_, -1)
                    cv2.putText(img, tx, (fx - tw // 2, int(t.y1) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5 * sc, (255, 255, 255), max(1, int(sc)), cv2.LINE_AA)
            if last_id is not None and np.isfinite(lx):                    # label at the last defender
                r_ = g[g.track_id == last_id].iloc[0]
                txt = f"last defender #{NUM.get(last_id, '')}: {lx:.1f} m from goal"
                (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.6 * sc, max(1, int(2 * sc)))
                org = (int((r_.x1 + r_.x2) / 2) - tw // 2, int(r_.y2) + int(38 * sc))
                org = (min(max(org[0], 6), Wv - tw - 6), min(org[1], Hv - int(10 * sc)))
                cv2.rectangle(img, (org[0] - 6, org[1] - th - 7), (org[0] + tw + 6, org[1] + 7), (20, 20, 20), -1)
                cv2.putText(img, txt, org, cv2.FONT_HERSHEY_SIMPLEX, 0.6 * sc, (0, 255, 255), max(1, int(2 * sc)), cv2.LINE_AA)
            # mini pitch (defending team's own goal on the left): players, defenders' line, last-defender line
            x0, y0 = Wv - mw - int(15 * sc), Hv - mh - int(15 * sc)
            ov = img.copy(); cv2.rectangle(ov, (x0, y0), (x0 + mw, y0 + mh), (40, 110, 40), -1); cv2.addWeighted(ov, 0.85, img, 0.15, 0, img)
            cv2.rectangle(img, (x0, y0), (x0 + mw, y0 + mh), (235, 235, 235), 1)
            cv2.line(img, (x0 + mw // 2, y0), (x0 + mw // 2, y0 + mh), (235, 235, 235), 1)
            if np.isfinite(lx):
                xx = int(x0 + lx * msc)
                for yy in range(y0, y0 + mh, 8):
                    cv2.line(img, (xx, yy), (xx, min(yy + 4, y0 + mh)), (0, 255, 255), 1)
            if defs is not None and len(defs) >= 2:
                mp = np.stack([x0 + defs.xa.values * msc, y0 + defs.ya.values * msc], 1).astype(np.int32)
                cv2.polylines(img, [mp], False, col, 2, cv2.LINE_AA)
            for t in g.itertuples():
                if not np.isfinite(t.pitch_x) or t.role == "referee":
                    continue
                mine = (not pd.isna(t.team)) and int(t.team) == k
                cv2.circle(img, (int(x0 + along_x(t.pitch_x, k) * msc), int(y0 + along_y(t.pitch_y, k) * msc)),
                           max(3, int(5 * sc)), col if mine else (200, 200, 200), -1, cv2.LINE_AA)
            b_ = (bt.ball_pitch_x.get(f, np.nan), bt.ball_pitch_y.get(f, np.nan))
            if np.isfinite(b_[0]):
                cv2.circle(img, (int(x0 + along_x(b_[0], k) * msc), int(y0 + along_y(b_[1], k) * msc)), max(2, int(3 * sc)), (0, 255, 255), -1)
        # live graph: last-defender distance from goal, with cursor
        ov = img.copy(); cv2.rectangle(ov, (gx, gy), (gx + gw, gy + gh), (25, 25, 25), -1); cv2.addWeighted(ov, 0.75, img, 0.25, 0, img)
        vals = series.values; lo_, hi_ = 0.0, max(55.0, np.nanmax(vals) + 3 if np.isfinite(vals).any() else 55)
        pts = [(int(gx + i / max(len(vals) - 1, 1) * gw), int(gy + gh - (v - lo_) / (hi_ - lo_) * gh))
               for i, v in enumerate(vals) if np.isfinite(v)]
        if len(pts) >= 2:
            cv2.polylines(img, [np.array(pts, np.int32)], False, col, 2, cv2.LINE_AA)
        cx_ = int(gx + (f - s0) / max(s1 - s0, 1) * gw)
        cv2.line(img, (cx_, gy), (cx_, gy + gh), (255, 255, 255), 1)
        cv2.putText(img, "last defender, m from goal", (gx + 6, gy + int(18 * sc)), cv2.FONT_HERSHEY_SIMPLEX, 0.45 * sc, (230, 230, 230), 1, cv2.LINE_AA)
        ov = img.copy(); cv2.rectangle(ov, (0, 0), (Wv, int(40 * sc)), (20, 20, 20), -1); cv2.addWeighted(ov, 0.7, img, 0.3, 0, img)
        state = "" if np.isfinite(lx) else "   [no line: not pressing]"
        cv2.putText(img, f"{mmss(f)}  {nm} defensive line | {m['kind'].replace('_', ' ')} | {m['note']}{state}", (int(12 * sc), int(27 * sc)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65 * sc, (255, 255, 255), max(1, int(sc)), cv2.LINE_AA)
        out.write(img)
    out.release()
    return path


CLIPS = []
if RENDER_CLIPS and MOMENTS:
    HC = _load_pickle(HOMOGRAPHY_PATH)
    cap = cv2.VideoCapture(VIDEO_PATH); assert cap.isOpened(), f"cannot open {VIDEO_PATH}"
    fps_v = cap.get(cv2.CAP_PROP_FPS) or FPS
    for m in MOMENTS:
        CLIPS.append({**m, "time": mmss(m["start"]), "clip": render_clip(m, HC, cap, fps_v)})
        print("  saved", CLIPS[-1]["clip"])
    cap.release()

## 9. 2D animations: top-down pitch + line-height timeline with a moving cursor
MP4 when ffmpeg is available to matplotlib, otherwise GIF.

In [ ]:
def animate(m):
    k, nm = m["team"], m["team_name"]
    s0, s1 = max(0, int(m["start"] - CLIP_PRE_S * FPS)), min(N_FRAMES - 1, int(m["end"] + CLIP_POST_S * FPS))
    frames = list(range(s0, s1 + 1, ANIM_STEP))
    rows = {f: g for f, g in live[(live.frame_idx >= s0) & (live.frame_idx <= s1)].groupby("frame_idx")}
    lh = LH.set_index(["team", "frame_idx"]).line_x_smooth
    fig = plt.figure(figsize=(11, 8.2)); gs = fig.add_gridspec(2, 1, height_ratios=[3, 1.2])
    ax, axt = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
    ax.add_patch(Rectangle((0, 0), L, W, fill=True, color="#3a7d44", zorder=0)); ax.add_patch(Rectangle((0, 0), L, W, fill=False, color="white"))
    ax.plot([52.5, 52.5], [0, W], color="white", lw=1)
    for x0 in (0, L - 16.5):
        ax.add_patch(Rectangle((x0, W / 2 - 20.16), 16.5, 40.32, fill=False, color="white", lw=1))
    ax.set_xlim(-3, L + 3); ax.set_ylim(W + 3, -3); ax.set_aspect("equal"); ax.axis("off")
    sc_ = {kk: ax.scatter([], [], s=180, color=RGB.get(TEAM_NAMES[kk]), edgecolor="white", zorder=3) for kk in TEAM_NAMES}
    txts = []
    ball = ax.scatter([], [], s=50, color="yellow", edgecolor="black", zorder=5)
    lines = {kk: ax.plot([], [], color=RGB.get(TEAM_NAMES[kk]), lw=2.5, ls="--", zorder=2)[0] for kk in TEAM_NAMES}
    ltxt = {kk: ax.text(0, -1.5, "", color=RGB.get(TEAM_NAMES[kk]), ha="center", fontsize=9, weight="bold") for kk in TEAM_NAMES}
    title = ax.set_title("")
    tt = np.arange(s0, s1 + 1)
    for kk in TEAM_NAMES:
        axt.plot(tt / FPS, [lh.get((kk, f), np.nan) for f in tt], color=RGB.get(TEAM_NAMES[kk]), lw=2, label=f"{TEAM_NAMES[kk]} line")
    cur = axt.axvline(s0 / FPS, color="black"); axt.set_ylabel("m from own goal"); axt.set_xlabel("match time (s)")
    axt.legend(loc="upper right", fontsize=8); axt.axvspan(m["start"] / FPS, m["end"] / FPS + 1e-3, color="orange", alpha=0.15)

    def upd(f):
        for t_ in txts: t_.remove()
        txts.clear()
        g = rows.get(f)
        for kk in TEAM_NAMES:
            if g is None:
                sc_[kk].set_offsets(np.empty((0, 2))); continue
            d = g[(g.team == kk).fillna(False)]
            sc_[kk].set_offsets(d[["pitch_x", "pitch_y"]].values)
            for r in d.itertuples():
                if NUM.get(r.track_id) is not None:
                    txts.append(ax.text(r.pitch_x, r.pitch_y, str(NUM[r.track_id]), color="white", fontsize=7,
                                        ha="center", va="center", zorder=4, weight="bold"))
            v = lh.get((kk, f), np.nan)
            if np.isfinite(v):
                xp = to_pitch_x(v, kk); lines[kk].set_data([xp, xp], [0, W]); ltxt[kk].set_position((xp, -1.5))
                ltxt[kk].set_text(f"{TEAM_NAMES[kk]} {v:.1f} m")
            else:
                lines[kk].set_data([], []); ltxt[kk].set_text("")
        b_ = (bt.ball_pitch_x.get(f, np.nan), bt.ball_pitch_y.get(f, np.nan))
        ball.set_offsets([b_] if np.isfinite(b_[0]) else np.empty((0, 2)))
        cur.set_xdata([f / FPS, f / FPS])
        title.set_text(f"{mmss(f)}   {nm} -- {m['kind'].replace('_', ' ')}: {m['note']}")
        return []

    an = animation.FuncAnimation(fig, upd, frames=frames, interval=1000 * ANIM_STEP / FPS, blit=False)
    if animation.writers.is_available("ffmpeg"):
        path = os.path.join(ANIM_DIR, f"{nm}_{m['kind']}_{m['start']}.mp4")
        an.save(path, writer=animation.FFMpegWriter(fps=FPS / ANIM_STEP, bitrate=2400))
    else:
        path = os.path.join(ANIM_DIR, f"{nm}_{m['kind']}_{m['start']}.gif")
        an.save(path, writer=animation.PillowWriter(fps=FPS / ANIM_STEP))
    plt.close(fig)
    return path


ANIMS = []
if RENDER_ANIMATIONS:
    for m in MOMENTS:
        ANIMS.append({**m, "animation": animate(m)})
        print("  saved", ANIMS[-1]["animation"])

## 10. Save tables + JSON report (for the LLM step)

In [ ]:
LH.to_parquet(os.path.join(OUT_DIR, "line_height_frames.parquet"), index=False)
if len(EV): EV.to_csv(os.path.join(OUT_DIR, "line_events.csv"), index=False, encoding="utf-8-sig")
DT.to_csv(os.path.join(OUT_DIR, "back_line_defenders.csv"), index=False, encoding="utf-8-sig")
report = {}
for k, nm in TEAM_NAMES.items():
    o = prs[prs.team == k]
    report[nm] = {
        "scope": "pressing frames only (no attacking, no passive defending, no set pieces"
                 + ("" if INCLUDE_COUNTER_PRESS else ", no counter-press") + ")",
        "back_line": [LABEL[t] for t in DEF[k]],
        "while_pressing": {"presses": int(o.press_seq.nunique()), "minutes": round(len(o) / FPS / 60, 2),
                           "last_defender_median_m": round(float(o.line_x.median()), 1) if len(o) else None,
                           "iqr_m": [round(float(o.line_x.quantile(.25)), 1), round(float(o.line_x.quantile(.75)), 1)] if len(o) else None,
                           "back_line_mean_median_m": round(float(o.line_mean_x.median()), 1) if len(o) else None,
                           "by_press_zone_median_m": o.groupby("block").line_x.median().round(1).to_dict(),
                           "by_situation_median_m": o.groupby("press_situation").line_x.median().round(1).to_dict(),
                           "flatness_m": round(float(o.flatness.median()), 1) if len(o) else None,
                           "width_m": round(float(o.width.median()), 1) if len(o) else None,
                           "gk_to_line_m": round(float(o.gk_to_line_m.median()), 1) if len(o) else None,
                           "line_to_midfield_m": round(float(o.line_to_mid_m.median()), 1) if len(o) else None},
        "events_during_presses": (EV[EV.team == k].groupby(["kind", "cause"]).size().unstack(fill_value=0).to_dict() if len(EV) else {}),
        "defenders": DT[DT.team == nm].drop(columns="team").round(2).to_dict("records"),
        "clips": [{k_: v for k_, v in c.items() if k_ != "team"} for c in CLIPS if c["team"] == k],
        "animations": [a["animation"] for a in ANIMS if a["team"] == k],
    }
with open(os.path.join(OUT_DIR, "defensive_line_report.json"), "w", encoding="utf-8") as fh:
    json.dump(report, fh, indent=1, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print(f"Saved to {OUT_DIR}: line_height_frames, line_events, back_line_defenders, defensive_line_report.json, "
      f"figures, {len(CLIPS)} clips, {len(ANIMS)} animations")